In [0]:
year = "2025"
rnd = dbutils.widgets.get("rnd")

In [0]:
import requests
import pandas as pd

url = f"https://api.jolpi.ca/ergast/f1/{year}/{rnd}/results/"
response = requests.get(url)
data = response.json()

df = pd.json_normalize(data)

In [0]:
df_spark = spark.createDataFrame(df)

if not spark.catalog.tableExists("`drive-to-survive`.default.api_response"):
    df_spark.write.mode("overwrite").saveAsTable("`drive-to-survive`.default.api_response")
else:
    df_spark.write.mode("overwrite").saveAsTable("`drive-to-survive`.default.api_response_temp")

In [0]:
%sql

MERGE INTO `drive-to-survive`.default.api_response AS target
USING `drive-to-survive`.default.api_response_temp AS source
ON target.`MRData.RaceTable.round` = source.`MRData.RaceTable.round`
WHEN MATCHED THEN
  UPDATE SET
    target.`MRData.xmlns` = source.`MRData.xmlns`,
    target.`MRData.series` = source.`MRData.series`,
    target.`MRData.url` = source.`MRData.url`,
    target.`MRData.limit` = source.`MRData.limit`,
    target.`MRData.offset` = source.`MRData.offset`,
    target.`MRData.total` = source.`MRData.total`,
    target.`MRData.RaceTable.season` = source.`MRData.RaceTable.season`,
    target.`MRData.RaceTable.Races` = source.`MRData.RaceTable.Races`
WHEN NOT MATCHED THEN
  INSERT (
    `MRData.xmlns`,
    `MRData.series`,
    `MRData.url`,
    `MRData.limit`,
    `MRData.offset`,
    `MRData.total`,
    `MRData.RaceTable.season`,
    `MRData.RaceTable.round`,
    `MRData.RaceTable.Races`
  )
  VALUES (
    source.`MRData.xmlns`,
    source.`MRData.series`,
    source.`MRData.url`,
    source.`MRData.limit`,
    source.`MRData.offset`,
    source.`MRData.total`,
    source.`MRData.RaceTable.season`,
    source.`MRData.RaceTable.round`,
    source.`MRData.RaceTable.Races`
  )